# 🏷️ The Price Is Right — Week 7 Exercise

## Improving QLoRA Fine-Tuning with Balanced Price-Bucket Sampling

**Author: Vagz1216 (Haqs12)**

---

### The Problem I'm Solving

In the instructor's Day 3/4 training notebook, the training data is loaded directly from
`ed-donner/items_prompts_lite` with no filtering — a random slice of 20,000 Amazon products.

The issue is that Amazon is heavily skewed toward cheap items. A random sample of 20,000
products will contain far more `$5 phone cases` and `$12 cables` than `$400 power tools` or
`$800 appliances`. When the fine-tuned model trains on this imbalanced data, it learns to
be excellent at guessing cheap prices — but it has seen so few expensive items that it
consistently underestimates them.

### My Solution: Price-Bucket Balanced Training Data

In my Week 6 exercise, I built a **price-bucket balancing algorithm** for OpenAI fine-tuning.
I'm applying that same idea here to the QLoRA open-source training pipeline.

I sort the training data into 4 price buckets and sample **equally** from each one:

| Bucket | Price Range | Items Selected |
|--------|-------------|----------------|
| Budget | \$0 – \$50 | 25% of training set |
| Mid-range | \$50 – \$150 | 25% of training set |
| Premium | \$150 – \$300 | 25% of training set |
| Luxury | \$300+ | 25% of training set |

The hypothesis is: **a model trained on a balanced price distribution will make more
accurate predictions across the full price range**, not just for cheap items.

> 🖥️ **Run in Google Colab with a free T4 GPU.**  
> Add your `HF_TOKEN` and `WANDB_API_KEY` to Colab Secrets before running.

---
## Step 1: Install Libraries

In [ ]:
# Pinned to the same versions as the instructor's Day 3/4 training notebook
!pip install -q --upgrade bitsandbytes==0.48.2 trl==0.25.1

# Pulling Ed's evaluate() and Tester utilities from the course repo
!wget -q https://raw.githubusercontent.com/ed-donner/llm_engineering/main/week7/util.py -O util.py

print("Libraries installed!")

In [ ]:
import os
import re
import random
import math
from collections import defaultdict
from datetime import datetime
from tqdm import tqdm
from google.colab import userdata
from huggingface_hub import login
import torch
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, set_seed
from datasets import load_dataset, Dataset, DatasetDict
from peft import LoraConfig, PeftModel
from trl import SFTTrainer, SFTConfig
import wandb
import matplotlib.pyplot as plt
from util import evaluate

print(f"PyTorch: {torch.__version__} | CUDA: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

---
## Step 2: Constants

I'm keeping `LITE_MODE = True` since I'm on a free T4 GPU.
The key difference from the instructor's notebook is that I will **not** use the
pre-formatted `items_prompts_lite` dataset directly.
Instead I load the raw `items_lite` dataset so I can access the price field
and apply my own balanced sampling before re-formatting.

In [ ]:
# Base open-source model — same as the instructor
BASE_MODEL = "meta-llama/Llama-3.2-3B"
PROJECT_NAME = "price"

# I load the raw items dataset (with price field) instead of pre-formatted prompts
# so I can apply price-bucket filtering before formatting
DATA_USER = "ed-donner"
RAW_DATASET = f"{DATA_USER}/items_lite"    # Has .price, .summary, .title fields
PROMPT_DATASET = f"{DATA_USER}/items_prompts_lite"  # Used for test evaluation only

# Keep within T4 GPU limits
LITE_MODE = True

# My HuggingFace username — update this to yours when running
HF_USER = "Haqs12"

# Run tracking
# --- Training / Model Loading Config ---
# By default, this notebook runs a fresh training job dynamically:
RUN_NAME = f"{datetime.now():%Y-%m-%d_%H.%M.%S}-lite-balanced"
PROJECT_RUN_NAME = f"{PROJECT_NAME}-{RUN_NAME}"
HUB_MODEL_NAME = f"{HF_USER}/{PROJECT_RUN_NAME}"

# 💡 NOTE FOR REVIEWERS: If you want to skip training and evaluate the EXACT model
# that achieved the $79.18 average error in the results below, comment out the 3 lines
# above and uncomment the 3 lines below to load my fine-tuned snapshot directly:
# RUN_NAME = "2026-03-11_11.38.39-lite-balanced"
# PROJECT_RUN_NAME = f"{PROJECT_NAME}-{RUN_NAME}"
# HUB_MODEL_NAME = "Haqs12/price-2026-03-11_11.38.39-lite-balanced"

# --- Training Hyperparameters (same as instructor's LITE_MODE settings) ---
EPOCHS = 1
BATCH_SIZE = 32
MAX_SEQUENCE_LENGTH = 128
GRADIENT_ACCUMULATION_STEPS = 1

# --- QLoRA Hyperparameters (same as instructor's LITE_MODE settings) ---
QUANT_4_BIT = True
LORA_R = 32
LORA_ALPHA = LORA_R * 2       # Standard convention: alpha = 2x rank
ATTENTION_LAYERS = ["q_proj", "v_proj", "k_proj", "o_proj"]
TARGET_MODULES = ATTENTION_LAYERS   # Targeting attention layers only (T4 memory limit)
LORA_DROPOUT = 0.1

# --- Optimizer & LR ---
LEARNING_RATE = 1e-4
WARMUP_RATIO = 0.01
LR_SCHEDULER_TYPE = 'cosine'
OPTIMIZER = "paged_adamw_32bit"

# Detect GPU capability
capability = torch.cuda.get_device_capability()
use_bf16 = capability[0] >= 8

# Logging
VAL_SIZE = 500
LOG_STEPS = 5
SAVE_STEPS = 100
LOG_TO_WANDB = True

print(f"GPU capability: {capability} → {'bfloat16' if use_bf16 else 'float16'}")
print(f"Model will be saved to HuggingFace as: {HUB_MODEL_NAME}")

---
## Step 3: Log In

In [ ]:
# HuggingFace — needed to download the raw dataset and push the fine-tuned model
hf_token = userdata.get('HF_TOKEN')
login(hf_token, add_to_git_credential=True)
print("Logged in to HuggingFace!")

# Weights & Biases — for tracking training loss curves
wandb_api_key = userdata.get('WANDB_API_KEY')
os.environ["WANDB_API_KEY"] = wandb_api_key
wandb.login()
os.environ["WANDB_PROJECT"] = PROJECT_NAME
os.environ["WANDB_LOG_MODEL"] = "false"
os.environ["WANDB_WATCH"] = "false"
print("Logged in to Weights & Biases!")

---
## Step 4: Load Raw Dataset & Apply Price-Bucket Balanced Sampling

This is the core improvement. Instead of feeding the QLoRA trainer a random slice
of the dataset, I first inspect every item's price, sort them into 4 buckets,
and pull an equal number from each bucket.

The `items_lite` dataset has a `price` field we can use directly for bucketing.
The instructor's `items_prompts_lite` skips this — it only has `prompt` and `completion`
strings, which makes price-based filtering much harder.

I then re-format the balanced items into `prompt`/`completion` pairs using the same
template the instructor uses (`What does this cost to the nearest dollar?...Price is $`).

In [ ]:
# Load the raw items dataset — has price, summary, title, category etc.
raw = load_dataset(RAW_DATASET)
raw_train = raw['train']
raw_val = raw['validation']

print(f"Raw training items: {len(raw_train):,}")
print(f"Raw validation items: {len(raw_val):,}")
print("\nSample item fields:", list(raw_train[0].keys()))

In [ ]:
# --- Price Bucket Definitions ---
# The same 4-bucket system I built in my Week 6 exercise.
# The goal is to force equal representation of cheap and expensive items.

def categorize_price(price):
    """Sort a price into one of 4 buckets."""
    if price < 50:
        return '$0-50'
    elif price < 150:
        return '$50-150'
    elif price < 300:
        return '$150-300'
    else:
        return '$300+'


def balance_by_price(dataset, items_per_bucket, seed=42):
    """
    Groups dataset items by price bucket and samples equally from each.
    Returns a shuffled, balanced list of HuggingFace dataset row dicts.
    """
    random.seed(seed)
    buckets = defaultdict(list)

    for item in dataset:
        price = item.get('price')
        # Skip items with no price or invalid price
        if price is None or not isinstance(price, (int, float)) or price <= 0:
            continue
        bucket = categorize_price(price)
        buckets[bucket].append(item)

    print("Price distribution BEFORE balancing:")
    for b in sorted(buckets.keys()):
        print(f"  {b}: {len(buckets[b]):,} items")

    balanced = []
    for b in sorted(buckets.keys()):
        sample_size = min(items_per_bucket, len(buckets[b]))
        sample = random.sample(buckets[b], sample_size)
        balanced.extend(sample)
        print(f"  Selected {sample_size} items from {b} bucket")

    random.shuffle(balanced)  # Shuffle so prices aren't grouped during training
    return balanced


# Apply balancing — 1,000 items per bucket = 4,000 total training items
# This is still well within T4 memory limits for LITE_MODE
ITEMS_PER_BUCKET = 1000

print("\n--- Balancing Training Data ---")
balanced_train_items = balance_by_price(raw_train, items_per_bucket=ITEMS_PER_BUCKET)
print(f"\nFinal balanced training set: {len(balanced_train_items):,} items")

print("\n--- Balancing Validation Data ---")
balanced_val_items = balance_by_price(raw_val, items_per_bucket=125)  # 125 per bucket = 500 val items
print(f"Final balanced validation set: {len(balanced_val_items):,} items")

In [ ]:
# Visualise the balanced training price distribution
# A balanced distribution should show roughly equal counts across the price range
train_prices = [item['price'] for item in balanced_train_items]

plt.figure(figsize=(12, 4))
plt.hist(train_prices, bins=50, color='steelblue', rwidth=0.8)
plt.title(f"Price Distribution of My Balanced Training Set ({len(train_prices):,} items)")
plt.xlabel("Price ($)")
plt.ylabel("Count")
plt.axvline(x=50, color='red', linestyle='--', alpha=0.7, label='Bucket boundaries')
plt.axvline(x=150, color='red', linestyle='--', alpha=0.7)
plt.axvline(x=300, color='red', linestyle='--', alpha=0.7)
plt.legend()
plt.tight_layout()
plt.show()

print(f"Price stats: min=${min(train_prices):.2f}, max=${max(train_prices):.2f}, "
      f"mean=${sum(train_prices)/len(train_prices):.2f}")

---
## Step 5: Format into Prompt/Completion Pairs

Now I convert my balanced `Item` dicts into the same `prompt`/`completion` string
format that the instructor's `SFTTrainer` expects.

The template is identical to what `items.py` generates — so the model will see
the same text structure it was designed for, just drawn from a more balanced
slice of the data.

In [ ]:
# These constants match exactly what the instructor defined in week7/pricer/items.py
QUESTION = "What does this cost to the nearest dollar?"
PREFIX = "Price is $"


def make_prompt_completion(item, tokenizer, max_tokens=128, include_answer=True):
    """
    Converts a raw item dict into the prompt/completion format expected by SFTTrainer.
    Mirrors the logic in the instructor's items.py make_prompts() method.
    """
    summary = item.get('summary') or item.get('title', '')

    # Truncate summary to max_tokens — same logic as the instructor's CUTOFF=110
    tokens = tokenizer.encode(summary, add_special_tokens=False)
    if len(tokens) > max_tokens:
        summary = tokenizer.decode(tokens[:max_tokens]).rstrip()

    prompt = f"{QUESTION}\n\n{summary}\n\n{PREFIX}"

    if include_answer:
        # Training/validation items include the answer: "Price is $299.00"
        completion = f"{round(item['price'])}.00"
    else:
        # Test items end at "Price is $" — model must predict the number
        completion = str(item['price'])

    return {"prompt": prompt, "completion": completion}


# Load the tokenizer now so we can use it for truncation during formatting
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("Formatting balanced training data...")
train_formatted = [make_prompt_completion(item, tokenizer, include_answer=True)
                   for item in tqdm(balanced_train_items)]

print("Formatting balanced validation data...")
val_formatted = [make_prompt_completion(item, tokenizer, include_answer=True)
                 for item in tqdm(balanced_val_items)]

# Convert to HuggingFace Dataset format — what SFTTrainer expects
train_dataset = Dataset.from_list(train_formatted)
val_dataset = Dataset.from_list(val_formatted)

print(f"\nTraining dataset: {len(train_dataset):,} items")
print(f"Validation dataset: {len(val_dataset):,} items")
print("\nSample formatted training item:")
print(train_dataset[0]['prompt'])
print("Completion:", train_dataset[0]['completion'])

---
## Step 6: Load the Base Model with 4-bit Quantization

Same as the instructor's Day 3/4 setup — load Llama 3.2 3B in 4-bit NF4 precision
to fit it within the T4's 16GB VRAM budget.

In [ ]:
# 4-bit quantization config — identical to the instructor's setup
if QUANT_4_BIT:
    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16 if use_bf16 else torch.float16,
        bnb_4bit_quant_type="nf4"
    )

# Load the frozen base model
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    device_map="auto",
)
base_model.generation_config.pad_token_id = tokenizer.pad_token_id

print(f"Base model loaded! Memory footprint: {base_model.get_memory_footprint() / 1e6:.1f} MB")

---
## Step 7: Configure QLoRA and Train

The LoRA and SFT configurations are the same as the instructor's LITE_MODE settings.
The only thing that has changed is **what data we're feeding in** — my balanced set
instead of a random slice. This isolates the data-quality improvement as the
independent variable in this experiment.

In [ ]:
# LoRA adapter configuration — same as instructor's LITE_MODE
lora_parameters = LoraConfig(
    lora_alpha=LORA_ALPHA,    # Scaling factor for the adapter updates
    lora_dropout=LORA_DROPOUT,
    r=LORA_R,                 # Rank of the low-rank decomposition (32 for LITE_MODE)
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=TARGET_MODULES,  # Only attention layers to save T4 memory
)

print(f"LoRA config: r={LORA_R}, alpha={LORA_ALPHA}, targets={TARGET_MODULES}")

In [ ]:
# Initialise WandB tracking run
if LOG_TO_WANDB:
    wandb.init(project=PROJECT_NAME, name=RUN_NAME)

# Training configuration — mirrors the instructor's LITE_MODE SFTConfig exactly
train_parameters = SFTConfig(
    output_dir=PROJECT_RUN_NAME,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    optim=OPTIMIZER,
    save_steps=SAVE_STEPS,
    save_total_limit=10,
    logging_steps=LOG_STEPS,
    learning_rate=LEARNING_RATE,
    weight_decay=0.001,
    fp16=not use_bf16,
    bf16=use_bf16,
    max_grad_norm=0.3,
    max_steps=-1,
    warmup_ratio=WARMUP_RATIO,
    group_by_length=True,
    lr_scheduler_type=LR_SCHEDULER_TYPE,
    report_to="wandb" if LOG_TO_WANDB else None,
    run_name=RUN_NAME,
    max_length=MAX_SEQUENCE_LENGTH,
    save_strategy="steps",
    hub_strategy="every_save",
    push_to_hub=True,       # Auto-saves to HuggingFace every SAVE_STEPS — crash-safe!
    hub_model_id=HUB_MODEL_NAME,
    hub_private_repo=True,
    eval_strategy="steps",
    eval_steps=SAVE_STEPS,
    dataset_text_field="prompt",
)

print(f"Training config ready. Will push checkpoints to: {HUB_MODEL_NAME}")

In [ ]:
# Wire everything together into the SFTTrainer
fine_tuning = SFTTrainer(
    model=base_model,
    train_dataset=train_dataset,      # My price-balanced training data
    eval_dataset=val_dataset,          # My price-balanced validation data
    peft_config=lora_parameters,
    args=train_parameters
)

# KICK OFF TRAINING!
# On a free T4 with 4,000 items, this should take roughly 10-20 minutes
fine_tuning.train()

# Push the final model to HuggingFace when done
fine_tuning.model.push_to_hub(PROJECT_RUN_NAME, private=True)
print(f"\nTraining complete! Model saved to: {HUB_MODEL_NAME}")

if LOG_TO_WANDB:
    wandb.finish()

---
## Step 8: Evaluate — Did the Balanced Training Data Help?

Now I test my price-balanced fine-tuned model against the same 200-item test set
the instructor uses, to directly compare my score with the baseline.

In [ ]:
# Load the standard test split for a fair apples-to-apples comparison
# with the instructor's baseline model score
prompt_ds = load_dataset(PROMPT_DATASET)
test = prompt_ds['test']
print(f"Test items: {len(test):,}")

In [ ]:
# Load my newly trained model for evaluation
# (If you're running evaluation after a crash, update HUB_MODEL_NAME to your repo)
fine_tuned_model = PeftModel.from_pretrained(base_model, HUB_MODEL_NAME)
print(f"Fine-tuned model loaded! Memory: {fine_tuned_model.get_memory_footprint() / 1e6:.1f} MB")

In [ ]:
def model_predict(item):
    """
    Standard inference function — identical to the instructor's Day 5 model_predict().
    Using the same greedy decoding so any improvement in score comes purely from
    the better-balanced training data, not a decoding trick.
    """
    inputs = tokenizer(item["prompt"], return_tensors="pt").to("cuda")
    with torch.no_grad():
        with torch.autocast("cuda", dtype=torch.float16):
            output_ids = fine_tuned_model.generate(**inputs, max_new_tokens=8)
    prompt_len = inputs["input_ids"].shape[1]
    generated_ids = output_ids[0, prompt_len:]
    return tokenizer.decode(generated_ids)


# Sanity check on 3 items
print("Sanity check:")
for i in range(3):
    pred = model_predict(test[i]).strip()
    actual = test[i]['completion']
    print(f"  Item {i+1}: Predicted '{pred}' | Actual: ${actual}")


In [ ]:
# Full evaluation across 200 unseen test items
print("Running full evaluation (200 test items)...")
set_seed(42)
evaluate(model_predict, test)

---
## Results & Reflection

| Model | Training Data | Avg Error ($) |
|---|---|---|
| Instructor's Baseline (random sample) | `items_prompts_lite` (20,000 items) | $65.40 |
| My Model (price-balanced sample) | `items_lite` balanced by bucket (4,000 items) | $79.18 |

### What I observed:

My fine-tuned model (trained on just 4,000 price-balanced items) achieved an average error of **$79.18**.

The results indicate that my model performed poorer when compared to the instructor's lite fine-tuned model ($65.40 vs $79.18).
However, there is an extremely important caveat: **the instructor's model trained on 20,000 items** — 5x more data than mine.

Despite training on a much smaller dataset, my model still **handily beat the Human baseline ($87.62)** and the **Base Llama 3.2 4-bit model ($110.72)**! (As shown in the instructor's general evaluation charts).

The key insight from this experiment is that **data quality matters**. A naive random sample of Amazon data is dominated by cheap items. By forcing the training set to see an equal number of items from each price tier, the model is exposed to a more representative version of the real world.

With time, I will try more approaches to see it perform better. For instance, scaling this balanced approach up to 20,000 items (which would require a paid GPU to fit the batch memory limits), I hypothesize it would beat the random-sample baseline entirely on expensive product accuracy.

This directly builds on the Week 6 data engineering lesson:
> *"Data Curation can be considered the less glamorous work of a Data Scientist.
> I say that's nonsense! This is where the science happens."* — Ed Donner

---
_Week 7 Capstone Exercise | Vagz1216 | Andela AI Engineering Bootcamp_
